#### <b>Installation Dependencies</b>
The understatapi contains historical records for the 6 largest European soccer leagues (English Premier League, La Liga, Ligue 1, Serie A, Bundesliga, and Russian Premier League) for every season since the 2014/15 season.

Run the following installation line in your command prompt to access the soccer data and use the other packages in this notebook:

``
pip install understatapi numpy pandas
``

If you're encountering any issues with understatapi, please refer to the following documentation: https://collinb9.github.io/understatAPI/index.html#

#### <b>Import Statements</b>

In [2]:
import numpy as np
import pandas as pd
from understatapi import UnderstatClient
# understat = UnderstatClient()

#### <b>API Endpoint Testing</b>

In [ ]:
# API Endpoint Testing
with UnderstatClient() as understat:
    team_match_data = understat.team(team="Manchester_United").get_match_data(season="2019")
    team_context_data = understat.team(team="Manchester_United").get_context_data(season="2019")
    team_data = understat.league(league="EPL")._get_data(season="2019")
    prem_2019_match_data = understat.league(league="EPL").get_match_data(season="2019")
    prem_2019_team_data = understat.league(league="EPL").get_team_data(season="2019")
    sample_id = prem_2019_match_data[0]["id"]
    sample_match_info = understat.match(match=sample_id).get_match_info()

len(prem_2019_match_data) # result is 380 matches (total matches across all clubs during a season)
len(prem_2019_team_data["71"]["history"]) # result is 38 matches (total matches for one club during a season)
len(prem_2019_team_data) # results is 20 teams (total teams competing for each season)
# prem_2019_match_data
# sample_match_info
prem_2019_team_data

In [ ]:
# API Endpoint Testing (cont.)
with UnderstatClient() as understat:
    cry_match_data_2026 = understat.team(team="Crystal_Palace").get_match_data(season="2026")
    league_match_data_2026 = understat.league(league="EPL").get_match_data(season="2026")
    league_team_data_cry = understat.league(league="EPL").get_team_data(season="2026")["78"]


cry_match_data_2026 # id = 78
league_match_data_2026
league_team_data_cry

#### <b>Data Consolidation</b>

Created the following pandas dataframe that captures descriptive information and performance statistics for all of the matches played by each club in the English Premier League from 2015 to 2026. The dataframe is titled `epl_match_data`.

Below are the columns included in `epl_match_data` along with their definitions.
1. Team — club name
2. Team_ID — Understat team identifier 
3. Match_ID — Understat match identifier
4. Opponent — opposing club
5. Home_or_Away — whether the team was home (h) or away (a)
6. Season — season start year
7. Result — match outcome for the team
8. Win_Percentage — Understat estimated probability of a win
9. Loss_Percentage — Understat estimated probability of a loss
10. Draw_Percentage — Understat estimated probability of a draw
11. Date — match date and kickoff time
12. Goals — goals scored by the team
13. Shots — total shots taken by the team
14. Shots_on_Target — shots on target by the team
15. Opp_Goals — goals scored by the opponent
16. Opp_Shots — total shots taken by the opponent
17. Opp_Shots_on_Target — opponent shots on target
18. xG — expected goals for the team
19. xGA — expected goals against
20. npxG — non-penalty expected goals for the team
21. npxGA — non-penalty expected goals against
22. npxGD — non-penalty expected goal differential, npxG - npxGA
23. Deep — deep completions by the team
24. Deep_Allowed — deep completions allowed to the opponent
25. Expected_Points — expected points for the match
26. Points — actual league points earned in the match
27. Total_Points — cummulative season points for the team

In [ ]:
# Collect information and statistics for each club's matches across each season from 2015 to 2025 (average download time -> 14 minutes)
match_rows = []

with UnderstatClient() as understat:
    for year in range(2015, 2026):
        match_info_cache = {}
        season_match_lookup = {}
        season_team_data = understat.league(league="EPL").get_team_data(season=f"{year}")
        season_match_data = understat.league(league="EPL").get_match_data(season=f"{year}")

        for game in season_match_data:
            season_match_lookup[(game["h"]["id"], game["datetime"])] = game["id"]
            season_match_lookup[(game["a"]["id"], game["datetime"])] = game["id"]

        for club, data in season_team_data.items():
            club_name = data["title"]
            club_ID = data["id"]
            history = data["history"]
            total_points = 0
            for match in history:
                home_or_away = match["h_a"]
                match_date = match["date"]
                match_id = season_match_lookup[(club_ID, match_date)]
                if match_id not in match_info_cache:
                    match_info_cache[match_id] = understat.match(match=match_id).get_match_info()
                match_info = match_info_cache[match_id]
                opponent = (
                    match_info[f"team_a"]
                    if home_or_away == "h"
                    else match_info[f"team_h"]
                )
                opponent_goals = (
                    match_info[f"a_goals"]
                    if home_or_away == "h"
                    else match_info[f"h_goals"]
                )
                opponent_shots = (
                    match_info[f"a_shot"]
                    if home_or_away == "h"
                    else match_info[f"h_shot"]
                )
                opponent_SOT = (
                    match_info[f"a_shotOnTarget"]
                    if home_or_away == "h"
                    else match_info[f"h_shotOnTarget"]
                )
                win_odds = (
                    match_info["h_w"] if home_or_away == "h" else match_info["h_l"]
                )
                lose_odds = (
                    match_info["h_l"] if home_or_away == "h" else match_info["h_w"]
                )
                draw_odds = match_info["h_d"]
                total_points += match["pts"]
                match_rows.append(
                    {
                        "Team": club_name,
                        "Team_ID": club_ID,
                        "Match_ID": match_id,
                        "Opponent": opponent,
                        "Home_or_Away": home_or_away,
                        "Season": year,
                        "Date": match_date,
                        "Win_Percentage": float(win_odds),
                        "Loss_Percentage": float(lose_odds),
                        "Draw_Percentage": float(draw_odds),
                        "Goals": int(match["scored"]),
                        "Shots": int(match_info[f"{home_or_away}_shot"]),
                        "Shots_on_Target": int(match_info[f"{home_or_away}_shotOnTarget"]),
                        "Opp_Goals": int(opponent_goals),
                        "Opp_Shots": int(opponent_shots),
                        "Opp_Shots_on_Target": int(opponent_SOT),
                        "xG": float(match["xG"]),
                        "xGA": float(match["xGA"]),
                        "npxG": float(match["npxG"]),
                        "npxGA": float(match["npxGA"]),
                        "npxGD": float(match["npxGD"]),
                        "Deep": float(match["deep"]),
                        "Deep_Allowed": float(match["deep_allowed"]),
                        "Expected_Points": float(match["xpts"]),
                        "Points": match["pts"],
                        "Total_Points": total_points,
                    }
                )

#### <b>Preliminary Data Cleaning</b>

In [ ]:
# Convert extracted data into a dataframe
epl_match_data = pd.DataFrame(match_rows)
epl_match_data["Result"] = np.select([epl_match_data["Goals"] > epl_match_data["Opp_Goals"], epl_match_data["Goals"] < epl_match_data["Opp_Goals"], epl_match_data["Goals"] == epl_match_data["Opp_Goals"]], ["Win", "Loss", "Draw"], default = "Unkown")
first_cols = ["Team", "Team_ID", "Match_ID", "Opponent", "Home_or_Away", "Season", "Date", "Result", "Win_Percentage", "Loss_Percentage", "Draw_Percentage"]
remaining_cols = [col for col in epl_match_data if col not in first_cols]
epl_match_data = epl_match_data[first_cols + remaining_cols]
epl_match_data

,Team,Team_ID,Match_ID,Opponent,Home_or_Away,Season,Date,Result,Win_Percentage,Loss_Percentage,...,xG,xGA,npxG,npxGA,npxGD,Deep,Deep_Allowed,Expected_Points,Points,Total_Points
0,Aston Villa,71,82,Bournemouth,a,2015,2015-08-08 18:00:00,Win,0.2958,0.3435,...,0.782253,0.876106,0.782253,0.876106,-0.093853,2.0,11.0,1.2481,3,3
1,Aston Villa,71,91,Manchester United,h,2015,2015-08-14 22:45:00,Loss,0.1056,0.4291,...,0.219118,0.660404,0.219118,0.660404,-0.441286,3.0,5.0,0.7821,0,3
2,Aston Villa,71,104,Crystal Palace,a,2015,2015-08-22 18:00:00,Loss,0.2174,0.4747,...,0.765278,1.238070,0.765278,1.238070,-0.472792,6.0,9.0,0.9601,0,3
3,Aston Villa,71,113,Sunderland,h,2015,2015-08-29 18:00:00,Draw,0.8747,0.0248,...,2.274060,0.368848,1.512890,0.368848,1.144042,12.0,3.0,2.7246,1,4
4,Aston Villa,71,129,Leicester,a,2015,2015-09-13 19:00:00,Loss,0.2446,0.4550,...,0.953181,1.349870,0.953181,1.349870,-0.396689,5.0,11.0,1.0342,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8355,Nottingham Forest,249,29115,Sunderland,a,2025,2026-04-24 19:00:00,Win,0.6401,0.1184,...,1.542080,0.614323,1.542080,0.614323,0.927757,3.0,13.0,2.1618,3,39
8356,Nottingham Forest,249,29122,Chelsea,a,2025,2026-05-04 15:00:00,Win,0.4074,0.3108,...,2.310190,2.171320,1.549020,1.410150,0.138870,4.0,12.0,1.5040,3,42
8357,Nottingham Forest,249,29134,Newcastle United,h,2025,2026-05-10 14:00:00,Draw,0.3112,0.4214,...,1.464340,1.650670,1.464340,1.650670,-0.186330,9.0,11.0,1.2010,1,43
8358,Nottingham Forest,249,29145,Manchester United,a,2025,2026-05-17 12:30:00,Loss,0.0591,0.8387,...,1.844180,4.285320,1.844180,4.285320,-2.441140,11.0,14.0,0.2795,0,43


View the table results at the following website to verify `season_performance` data:
https://www.premierleague.com/en/tables/premier-league/2026-27

In [21]:
# Verifying dataset results based on match and season information
match_sample = understat.match(match = str(82)).get_match_info()
match_data_sample = epl_match_data[epl_match_data["Match_ID"] == str(82)]
len(epl_match_data) # should equal 8360 (760 matches across 20 teams (including duplicates) * 11 seasons)
epl_match_data.groupby(["Season", "Team"]).size() # should equal 220 (20 teams * 11 seasons)
epl_match_data.groupby(["Season", "Match_ID"]).size().value_counts() # should equal 2 for the 4180 matches across 11 seasons
season_performance = epl_match_data.groupby(["Season", "Team"])["Total_Points"].max() # should reflect the total points achieved by each team at the end of each season
season_performance = season_performance.reset_index()
season_performance = season_performance.sort_values(by=["Season", "Total_Points"], ascending=[True, False])

#### <b>Data Export</b>
Run the following line to save the premier league match data and the end of season results to an Excel file on your device.

In [ ]:
with pd.ExcelWriter("epl_match_data.xlsx") as writer:
    epl_match_data.to_excel(writer, sheet_name = "Match Data", index = False)
    season_performance.to_excel(writer, sheet_name = "Season Results Data", index = False)